<a href="https://colab.research.google.com/github/DanielKuzma/MEDICA/blob/main/MEDICA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Zainstaluj Flaska, jeśli środowisko go nie posiada (w Colab jest domyślnie, ale dla pewności)
!pip install Flask Werkzeug

import sqlite3
import threading
from flask import Flask, request, render_template_string, redirect, url_for, session, flash
from werkzeug.security import generate_password_hash, check_password_hash

app = Flask(__name__)
app.secret_key = 'medica_super_sekretny_klucz' # Niezbędne do obsługi sesji i flash messages

# --- INICJALIZACJA BAZY DANYCH ---
def init_db():
    conn = sqlite3.connect('medica.db')
    c = conn.cursor()
    c.execute('''
        CREATE TABLE IF NOT EXISTS pacjenci (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            imie TEXT NOT NULL,
            nazwisko TEXT NOT NULL,
            pesel TEXT UNIQUE NOT NULL,
            telefon TEXT,
            email TEXT UNIQUE NOT NULL,
            haslo TEXT NOT NULL
        )
    ''')
    conn.commit()
    conn.close()

# --- SZABLONY HTML (CSS + STRUKTURA) ---
# Wspólny styl nawiązujący do mockupów
BASE_CSS = """
<style>
    body { font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background-color: #f4f6fa; color: #333; display: flex; justify-content: center; padding-top: 50px; margin: 0; }
    .container { background: #ffffff; padding: 40px; border-radius: 12px; box-shadow: 0 10px 25px rgba(0,0,0,0.05); width: 100%; max-width: 600px; }
    .card-container { max-width: 800px; }
    h2 { color: #222; margin-bottom: 5px; font-size: 22px; }
    .subtitle { color: #888; font-size: 13px; margin-bottom: 30px; }
    .logo { color: #5a189a; font-weight: bold; font-size: 18px; float: right; }
    .form-row { display: flex; gap: 20px; margin-bottom: 15px; }
    .form-group { flex: 1; display: flex; flex-direction: column; }
    label { font-size: 12px; font-weight: 600; margin-bottom: 8px; color: #444; }
    input[type="text"], input[type="email"], input[type="password"] { padding: 12px; border: 1px solid #e1e4e8; border-radius: 8px; background-color: #f9fafc; font-size: 14px; outline: none; }
    input[type="text"]:focus, input[type="email"]:focus, input[type="password"]:focus { border-color: #5a189a; }
    .btn-row { display: flex; gap: 15px; margin-top: 30px; }
    .btn { padding: 12px; border: none; border-radius: 8px; cursor: pointer; font-weight: bold; text-align: center; font-size: 14px; flex: 1; text-decoration: none; display: flex; justify-content: center; align-items: center;}
    .btn-primary { background-color: #5a189a; color: white; }
    .btn-secondary { background-color: #f0f0f5; color: #555; }
    .btn-outline { background-color: transparent; border: 2px solid #5a189a; color: #5a189a; }
    .alert { padding: 10px; background-color: #ffcccc; color: #cc0000; border-radius: 5px; margin-bottom: 15px; font-size: 13px; }

    /* Style specyficzne dla karty pacjenta */
    .section-card { border: 1px solid #eee; border-radius: 10px; padding: 25px; margin-bottom: 20px; background: white;}
    .section-title { color: #5a189a; font-size: 16px; font-weight: bold; margin-bottom: 20px; display: flex; align-items: center; gap: 10px; }
    .readonly-input { background-color: #f9fafc; color: #666; pointer-events: none; }
    textarea { width: 100%; height: 80px; padding: 12px; border: 1px solid #e1e4e8; border-radius: 8px; background-color: #f9fafc; resize: none; box-sizing: border-box; margin-bottom: 15px;}
</style>
"""

TPL_REGISTER = BASE_CSS + """
<div class="container">
    <div class="logo">⚕️ Medica+</div>
    <h2>Dodawanie nowego pacjenta</h2>
    <div class="subtitle">Wprowadź dane podstawowe, aby utworzyć kartę w systemie.</div>

    {% with messages = get_flashed_messages() %}
      {% if messages %}
        <div class="alert">{{ messages[0] }}</div>
      {% endif %}
    {% endwith %}

    <form method="POST">
        <div class="form-row">
            <div class="form-group">
                <label>Imię</label>
                <input type="text" name="imie" required placeholder="np. Daniel">
            </div>
            <div class="form-group">
                <label>Nazwisko</label>
                <input type="text" name="nazwisko" required placeholder="np. Kuzma">
            </div>
        </div>
        <div class="form-row">
            <div class="form-group">
                <label>PESEL <span style="color:#a881e6; font-size:10px;">UNIQUE</span></label>
                <input type="text" name="pesel" required placeholder="11 cyfr">
            </div>
        </div>
        <div class="form-row">
            <div class="form-group">
                <label>Telefon</label>
                <input type="text" name="telefon" required placeholder="np. 123456789">
            </div>
            <div class="form-group">
                <label>E-mail (Login)</label>
                <input type="email" name="email" required placeholder="daniel@gmail.com">
            </div>
        </div>
        <div class="form-row">
            <div class="form-group">
                <label>Hasło do systemu</label>
                <input type="password" name="haslo" required placeholder="••••••••">
                <div style="font-size:10px; color:#999; margin-top:5px;">Twoje hasło musi zawierać od 8 do 12 znaków.</div>
            </div>
        </div>
        <div class="btn-row">
            <button type="submit" class="btn btn-primary" style="flex: 2;">Zapisz pacjenta w bazie</button>
            <a href="/login" class="btn btn-secondary">Anuluj / Zaloguj</a>
        </div>
    </form>
</div>
"""

TPL_LOGIN = BASE_CSS + """
<div class="container" style="max-width: 400px;">
    <div class="logo">⚕️ Medica+</div>
    <h2>Logowanie do systemu Medica+</h2>
    <br>

    {% with messages = get_flashed_messages() %}
      {% if messages %}
        <div class="alert">{{ messages[0] }}</div>
      {% endif %}
    {% endwith %}

    <form method="POST">
        <div class="form-group" style="margin-bottom: 20px;">
            <label>Adres e-mail lub Login</label>
            <input type="email" name="email" required placeholder="podaj login">
        </div>
        <div class="form-group" style="margin-bottom: 10px;">
            <label>Hasło</label>
            <input type="password" name="haslo" required placeholder="podaj hasło">
        </div>

        <div style="font-size:12px; color:#555; margin-bottom: 25px; display:flex; align-items:center;">
            <input type="checkbox" style="margin-right: 8px;"> Zapamiętaj mnie na tym urządzeniu.
        </div>

        <div class="btn-row" style="margin-top: 0;">
            <button type="submit" class="btn btn-primary">Zaloguj</button>
            <button type="button" class="btn btn-outline">Odzyskaj hasło</button>
        </div>
        <div style="text-align:center; margin-top: 20px;">
            <a href="/register" style="font-size: 12px; color: #5a189a;">Nie masz konta? Zarejestruj się</a>
        </div>
    </form>
</div>
"""

TPL_CARD = BASE_CSS + """
<div class="container card-container">
    <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 20px;">
        <h2 style="margin:0;">Interfejs karty pacjenta:</h2>
        <div>
            <span class="logo" style="float:none; margin-right: 20px;">⚕️ Medica+</span>
            <a href="/logout" class="btn btn-secondary" style="padding: 8px 15px; display:inline-block; width:auto;">Wyloguj</a>
        </div>
    </div>

    <div class="section-card">
        <div class="section-title">📄 Dane podstawowe</div>
        <div class="form-row">
            <div class="form-group"><label>Imię</label><input type="text" class="readonly-input" value="{{ pacjent['imie'] }}" readonly></div>
            <div class="form-group"><label>Nazwisko</label><input type="text" class="readonly-input" value="{{ pacjent['nazwisko'] }}" readonly></div>
        </div>
        <div class="form-row">
            <div class="form-group"><label>PESEL</label><input type="text" class="readonly-input" value="{{ pacjent['pesel'] }}" readonly></div>
        </div>
        <div class="form-row">
            <div class="form-group"><label>Telefon</label><input type="text" class="readonly-input" value="{{ pacjent['telefon'] }}" readonly></div>
            <div class="form-group"><label>E-mail</label><input type="text" class="readonly-input" value="{{ pacjent['email'] }}" readonly></div>
        </div>
    </div>

    <div class="section-card">
        <div class="section-title">📅 Historia wizyt</div>
        <div class="form-group">
            <label>Wybierz termin wizyty</label>
            <input type="text" class="readonly-input" value="Brak wizyt w systemie" readonly>
        </div>
    </div>

    <div class="section-card">
        <div class="section-title">⚕️ Dokumentacja medyczna</div>
        <label>Wywiad lekarski</label>
        <textarea readonly class="readonly-input"></textarea>

        <label>Diagnoza</label>
        <textarea readonly class="readonly-input"></textarea>

        <label>Zalecenia</label>
        <textarea readonly class="readonly-input"></textarea>
    </div>
</div>
"""

# --- ROUTING I LOGIKA APLIKACJI ---

@app.route('/')
def index():
    return redirect(url_for('login'))

@app.route('/register', methods=['GET', 'POST'])
def register():
    if request.method == 'POST':
        imie = request.form['imie']
        nazwisko = request.form['nazwisko']
        pesel = request.form['pesel']
        telefon = request.form['telefon']
        email = request.form['email']
        haslo = generate_password_hash(request.form['haslo']) # Hashowanie hasła dla bezpieczeństwa

        try:
            conn = sqlite3.connect('medica.db')
            c = conn.cursor()
            c.execute("INSERT INTO pacjenci (imie, nazwisko, pesel, telefon, email, haslo) VALUES (?, ?, ?, ?, ?, ?)",
                      (imie, nazwisko, pesel, telefon, email, haslo))
            conn.commit()
            conn.close()
            flash('Pacjent został dodany. Możesz się zalogować.')
            return redirect(url_for('login'))
        except sqlite3.IntegrityError:
            flash('Błąd: Konto z tym adresem E-mail lub numerem PESEL już istnieje!')
            return redirect(url_for('register'))

    return render_template_string(TPL_REGISTER)

@app.route('/login', methods=['GET', 'POST'])
def login():
    if request.method == 'POST':
        email = request.form['email']
        haslo = request.form['haslo']

        conn = sqlite3.connect('medica.db')
        conn.row_factory = sqlite3.Row
        c = conn.cursor()
        c.execute("SELECT * FROM pacjenci WHERE email = ?", (email,))
        pacjent = c.fetchone()
        conn.close()

        if pacjent and check_password_hash(pacjent['haslo'], haslo):
            session['pacjent_id'] = pacjent['id']
            return redirect(url_for('karta'))
        else:
            flash('Nieprawidłowy e-mail lub hasło.')
            return redirect(url_for('login'))

    return render_template_string(TPL_LOGIN)

@app.route('/karta')
def karta():
    if 'pacjent_id' not in session:
        return redirect(url_for('login'))

    conn = sqlite3.connect('medica.db')
    conn.row_factory = sqlite3.Row
    c = conn.cursor()
    c.execute("SELECT * FROM pacjenci WHERE id = ?", (session['pacjent_id'],))
    pacjent = c.fetchone()
    conn.close()

    return render_template_string(TPL_CARD, pacjent=pacjent)

@app.route('/logout')
def logout():
    session.pop('pacjent_id', None)
    return redirect(url_for('login'))

# --- URUCHOMIENIE W GOOGLE COLAB ---
if __name__ == '__main__':
    init_db() # Tworzy bazę przy starcie

    # Uruchamiamy aplikację w tle (wątek), aby nie blokować komórki Colaba
    threading.Thread(target=app.run, kwargs={'host': '0.0.0.0', 'port': 5000, 'debug': False, 'use_reloader': False}).start()

    print("Aplikacja uruchamia się...")

    # Wykorzystujemy wbudowaną funkcję Colaba, by wystawić port na zewnątrz
    from google.colab import output
    output.serve_kernel_port_as_window(5000)
    print("Kliknij w link wygenerowany powyżej, aby otworzyć aplikację Medica+!")

Aplikacja uruchamia się...
Try `serve_kernel_port_as_iframe` instead. 
 * Serving Flask app '__main__'
 * Debug mode: off


<IPython.core.display.Javascript object>

Kliknij w link wygenerowany powyżej, aby otworzyć aplikację Medica+!


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.
